# M12.5.2 — X-only joint NPE: CPU analysis

Unexecuted analysis of **finished scalar outputs only**. Read `docs/experiments/M12_5_2_NPE_X_only.md` for the fixed hypothesis, support approximation, commands, domains and thresholds. No X/checkpoint loading, GPU, training, posterior sampling or waveform generation occurs here.

Primary: seed 123, historical held-out **validation environment mixture**. Seed 124 is fixed sensitivity, not a selectable competitor. Validation selection prevents confirmatory interpretation. Calibration/test remain untouched. Average learned variance is not automatically Bayes risk; B and irreducible fractions remain **UNRESOLVED**.


In [ ]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
from IPython.display import display

# Launch from repository root or this notebook's directory.
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/paths.py').is_file():
    if ROOT == ROOT.parent:
        raise RuntimeError('Cannot locate repository root')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.paths import resolve_data_root
ANALYSIS = resolve_data_root() / 'results/M11_final_G1_500k/M12_5_2_NPE/npe_analysis_v1'
GROUP = 'all'  # Select a predeclared region from the table; no threshold fitting.

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
receipt = json.loads((ANALYSIS / 'analysis.json').read_text())
assert receipt['complete']
tables = {}
for name, expected in receipt['tables'].items():
    path = ANALYSIS / (name + '.csv')
    assert sha256(path) == expected, f'Changed table: {name}'
    tables[name] = pd.read_csv(path)
assert sha256(ROOT / 'scripts/m12/analyze_m12_5_2_npe.py') == receipt['code_sha256']
display(pd.DataFrame(list(receipt['receipts'].items()), columns=['inference receipt path', 'SHA256']))
print(receipt['answers']['domain'])
print(receipt['answers']['selection'])


## A. Paired estimator efficiency

Positive delta favors NPE. Physical MSE uses squared solar masses or squared dimensionless spin; standardized MSE uses the historical train scaler. Simultaneous global intervals condition on observed environment counts. Region intervals form another multiplicity family; unresolved bootstrap tails and sparse regions cannot support decisions. `assessment` requires resolved source/MC precision, budget stability and consistent direction at the second seed; it does **not** certify conditional posterior accuracy.


In [ ]:
display(tables['assessment'])
display(tables['risk'].query('group == @GROUP'))
display(tables['environment_sensitivity'])
print('Available regions:', tables['risk']['group'].unique().tolist())


## Numerical and approximation sensitivity

Inspect MCSE in `risk`/`information`, nested sampling budgets and independent training seeds separately. Direct draws are IID only from the fitted density. Similar seeds are not an independent exact posterior validation. No finite-data/capacity approximation bound is available. All reported support violations must be zero; boundary-adjusted quantized labels remain an explicit approximation.


In [ ]:
display(tables['sampling_budget'])
display(tables['seed_sensitivity'].query('group == @GROUP'))
display(tables['support'])


## B. Posterior validation and information-limit consistency

B remains **UNRESOLVED**. Compare average posterior variance and empirical mean MSE with their uncertainty; equality of averages is insufficient. Marginal coverage/ranks can pass for the prior. Joint density ranks and energy scores add diagnostics, not a proof. Truth/SNR/environment-selected coverage is not coverage conditional on X and need not be nominal even for an exact posterior. There is no authorized posterior-predictive simulator or independently verified posterior reference.


In [ ]:
display(tables['information'].query('group == @GROUP'))
display(tables['coverage'].query('group == @GROUP'))
display(tables['informativeness'].query('group == @GROUP'))
rank_table = tables['ranks'].query('group == @GROUP')
display(rank_table.pivot(index='bin', columns='label', values='count'))
display(rank_table[['label', 'ks_distance', 'simultaneous_dkw_bound']].drop_duplicates())


## C. Avoidable risk and unresolved irreducible fraction

A positive resolved fraction below is the demonstrated reduction relative to this frozen M11 predictor, with paired uncertainty. Do not interpret the remaining fraction as irreducible, or rename learned variance/M11-MSE as an estimated irreducible fraction. The latter is **UNRESOLVED** for every target and region.


In [ ]:
columns = ['group','label','demonstrated_avoidable_fraction','avoidable_fraction_ci_low',
           'avoidable_fraction_ci_high','irreducible_fraction','mc_resolved','bootstrap_tail_resolved']
display(tables['risk'].query("group == @GROUP and units == 'physical'")[columns])


## D. Regression toward the mean

Compare OLS prediction-versus-truth slopes, intercepts and signed train-Q10/Q90 biases. Tail cuts are train-only. A slope below one and residual tail bias in an accurately validated posterior mean can reflect optimal Bayesian shrinkage; here approximation and environment shift remain alternative explanations. Report magnitudes rather than treating every slope below one as a model defect.


In [ ]:
display(tables['metrics'].query('group == @GROUP'))


## E. Evidence-based next intervention

A resolved same-X improvement supports investigating estimator training/capacity, without determining the cause or authorizing a new architecture. This experiment contains no matched D/U control and therefore cannot establish preprocessing information loss. Uncertainty-aware deployment needs stronger conditional validation. If precision/approximation is unresolved, do not choose an intervention from apparent regression alone.


In [ ]:
for question, answer in receipt['answers'].items():
    print(question, ':', answer, '\n')


## Final scientific record (complete after inspecting results)

- **A:** per-target supported risk improvement / equivalence / unresolved, with intervals, MC and seed/budget sensitivity. Posterior accuracy remains a separate question.
- **B:** **UNRESOLVED** in this experiment; list the observed diagnostic failures and missing independent/conditional/predictive evidence.
- **C:** report risk reduction fractions and bounds by target/region; irreducible fractions **UNRESOLVED**.
- **D:** record M11 and NPE slopes and signed tail biases, plus regions with persistent shrinkage.
- **E:** record the limited intervention conclusion justified above; do not infer preprocessing loss from X-only risks.

No values or outputs have been fabricated. Future confirmatory evaluation requires a reviewed validation lock; no test or calibration results enter this notebook.


## F. Comparación directa M11–NPE sobre las mismas observaciones

Estas figuras **no modifican la selección de checkpoint ni los contrastes predefinidos**. Abren únicamente el archivo comprimido de resúmenes posteriores (no `X`, ni checkpoints, ni `cal/test`) y verifican su SHA256 frente al recibo de inferencia. Se utiliza exclusivamente la semilla primaria 123 para las curvas; la sensibilidad a 124 se recoge en las tablas anteriores.

Las curvas \(\mathbb E[\hat\theta\mid\theta_{m true}]\) son **diagnósticos descriptivos por estratos definidos usando la verdad**: no representan el posterior bayesiano condicionado a \(X\). Una contracción hacia la media puede persistir incluso en el estimador óptimo bajo pérdida cuadrática.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Identify the primary inference receipt, always seed 123.
primary_dir = None
for candidate in receipt['receipts']:
    d = Path(candidate)
    cr = json.loads((d / 'complete.json').read_text())
    if cr['training_seed'] == 123:
        primary_dir = d
        assert sha256(d / 'complete.json') == receipt['receipts'][candidate]
        assert cr['split'] == 'val' and cr['complete']
        break
assert primary_dir is not None, 'Cannot locate seed 123 inference'
primary_receipt = json.loads((primary_dir / 'complete.json').read_text())
posterior_path = primary_dir / 'posterior.npz'
assert sha256(posterior_path) == primary_receipt['archive_sha256'], 'Posterior archive was modified'
with np.load(posterior_path, allow_pickle=False) as z:
    data = {k: z[k] for k in ('idx_val','source_id','environment','snr','y_physical','y_mean','y_std',
                              'm11_pred_val','mean','variance','quantiles','train_label_cuts',
                              'prior_variance','draws')}
assert len(data['idx_val']) == 40000
assert np.array_equal(data['idx_val'], np.arange(400000,440000))
labels = ['chirp_mass','total_mass','chi_eff']
y = data['y_physical'].astype(float)
m11 = data['m11_pred_val'].astype(float) * data['y_std'] + data['y_mean']
npe = data['mean'].astype(float)
snr = data['snr'].astype(float)
assert y.shape == m11.shape == npe.shape == (40000,3)
assert np.isfinite(y).all() and np.isfinite(npe).all()
print('Primary inference:', primary_dir, '| draws per row:', np.unique(data['draws']))

### F1. Regresión hacia la media: valores verdaderos frente a predicciones

Se representan nubes de puntos reducidas de forma reproducible, con curvas de media de la predicción por **veintiles de valor verdadero**. Los cortes se determinan con los valores de verdad de validation **sólo para visualización**, no para seleccionar modelos ni declarar significación estadística. Las colas inferenciales formales siguen siendo las Q10/Q90 predeclaradas en `metrics.csv`.

In [ ]:
rng = np.random.default_rng(125252)
subset = np.sort(rng.choice(len(y), size=min(3500,len(y)), replace=False))
fig, axes = plt.subplots(1, 3, figsize=(17,4.7))
for j, ax in enumerate(axes):
    ax.scatter(y[subset,j],m11[subset,j],s=4,alpha=.15,label='M11')
    ax.scatter(y[subset,j],npe[subset,j],s=4,alpha=.15,label='NPE posterior mean')
    low=min(y[:,j].min(),m11[:,j].min(),npe[:,j].min())
    high=max(y[:,j].max(),m11[:,j].max(),npe[:,j].max())
    ax.plot([low,high],[low,high],ls='--',c='k',lw=1,label='Identity')
    cuts=np.quantile(y[:,j],np.linspace(0,1,21)); cuts=np.unique(cuts)
    bins=np.clip(np.searchsorted(cuts,y[:,j],side='right')-1,0,len(cuts)-2)
    for prediction,name,marker in ((m11,'M11','o'),(npe,'NPE','s')):
        xy=[(np.mean(y[bins==b,j]),np.mean(prediction[bins==b,j])) for b in range(len(cuts)-1) if (bins==b).sum()>=50]
        if xy:
            xx,yy=np.array(xy).T
            ax.plot(xx,yy,marker=marker,ms=3,lw=1.6,label=f'{name} binned mean')
    ax.set(xlabel=f'True {labels[j]}',ylabel='Predicted',title=labels[j]);ax.legend(fontsize=7)
fig.suptitle('Regression toward the mean — validation, descriptive')
fig.tight_layout()
plt.show()

### F2. Error por SNR y por colas físicas

Los gráficos de error comparan las mismas observaciones y no deben confundirse con cotas del riesgo Bayes. Los cortes de SNR son los fijados en el protocolo. Los cortes Q10/Q90 proceden del train congelado.

In [ ]:
snr_groups=[('10–15',(snr>=10)&(snr<15)),('15–20',(snr>=15)&(snr<20)),('20–25',(snr>=20)&(snr<25))]
fig,axes=plt.subplots(1,3,figsize=(15,4))
for j,ax in enumerate(axes):
    for pred,name,shift in ((m11,'M11',-.16),(npe,'NPE',.16)):
        rmse=[np.sqrt(np.mean((pred[m,j]-y[m,j])**2)) if m.any() else np.nan for _,m in snr_groups]
        ax.bar(np.arange(3)+shift,rmse,width=.32,label=name)
    ax.set_xticks(range(3),[x[0] for x in snr_groups])
    ax.set(xlabel='Target network SNR',ylabel='RMSE',title=labels[j]);ax.legend()
fig.tight_layout();plt.show()

fig,axes=plt.subplots(1,3,figsize=(15,4))
for j,ax in enumerate(axes):
    cutlo,cutup=data['train_label_cuts'][:,j]
    masks=[y[:,j]<=cutlo,(y[:,j]>cutlo)&(y[:,j]<cutup),y[:,j]>=cutup]
    for pred,name,shift in ((m11,'M11',-.16),(npe,'NPE',.16)):
        bias=[np.mean(pred[m,j]-y[m,j]) if m.any() else np.nan for m in masks]
        ax.bar(np.arange(3)+shift,bias,width=.32,label=name)
    ax.axhline(0,color='k',lw=.8);ax.set_xticks(range(3),['≤train Q10','center','≥train Q90'])
    ax.set(xlabel='Truth stratum',ylabel='Signed bias (prediction − truth)',title=labels[j]);ax.legend()
fig.tight_layout();plt.show()

### F3. Varianza posterior y errores de la media, por SNR

Para un posterior exacto, la ley de la varianza condicional implica igualdad **en promedio sobre la distribución que define el posterior** entre \(\mathrm{Var}(Y\mid X)\) y el error cuadrático de su media. La comparación siguiente sólo verifica una consistencia agregada sobre la mezcla de validation. No constituye validación condicional de \(q_\phi(\theta\mid X)\).

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
for j,ax in enumerate(axes):
    avgvar=[np.mean(data['variance'][m,j]) if m.any() else np.nan for _,m in snr_groups]
    err=[np.mean((npe[m,j]-y[m,j])**2) if m.any() else np.nan for _,m in snr_groups]
    ax.bar(np.arange(3)-.16,avgvar,width=.32,label='Mean learned posterior variance')
    ax.bar(np.arange(3)+.16,err,width=.32,label='NPE posterior-mean MSE')
    ax.set_xticks(range(3),[x[0] for x in snr_groups]);ax.set(title=labels[j],ylabel='Squared physical units',xlabel='Target network SNR')
    ax.legend(fontsize=8)
fig.tight_layout();plt.show()

### F4. Cobertura por SNR (diagnóstico; no garantía condicional)

Las bandas son intervalos centrales aprendidos y no intervalos de confianza para \(\theta\) en el sentido frecuentista. La cobertura seleccionada por **SNR latente** no tiene por qué ser nominal incluso para el posterior exacto condicionado solamente a \(X\).

In [ ]:
qidx={.5:(3,5),.8:(2,6),.9:(1,7),.95:(0,8)}
fig,axes=plt.subplots(1,3,figsize=(15,4))
for j,ax in enumerate(axes):
    for level in (.5,.8,.9,.95):
        lo,hi=qidx[level]
        cov=[np.mean((y[m,j]>=data['quantiles'][m,lo,j])&(y[m,j]<=data['quantiles'][m,hi,j])) if m.any() else np.nan for _,m in snr_groups]
        ax.plot(range(3),cov,marker='o',label=f'{int(level*100)}% nominal')
    ax.set(xticks=range(3),xticklabels=[s[0] for s in snr_groups],ylim=(0,1.03),xlabel='Target network SNR',ylabel='Empirical coverage',title=labels[j]);ax.legend(fontsize=8)
fig.tight_layout();plt.show()

### F5. Registro de conclusiones sin extrapolaciones

- **Demostración empírica de mejora:** exige comparar intervalos emparejados, sensibilidad a semilla y Monte Carlo con los criterios de `assessment.csv`; validation sigue siendo exploratoria por selección.
- **Información irreducible:** **no identificada** por las figuras, ni por rangos/coverage agregados. No usar `average_posterior_variance` como cota de Bayes sin validación independiente.
- **Regresión a la media:** describir su cambio por parámetro y colas físicas; no interpretar la pendiente como prueba de error evitable.
- **Siguiente decisión:** basarse en evidencia confirmatoria y en un estudio específico del límite informativo, no en una comparación informal de modelos.